# Constructing the Transformer Block

In the last notebook, we built the `CausalSelfAttention` mechanism, the core "communication" primitive of a GPT model. But that's just one part of the puzzle. To build a powerful and trainable deep network, we need to wrap that attention mechanism in a standardized structure called a **Transformer Block**.

By the end of this notebook, you will understand the critical roles of Layer Normalization and residual connections. You will see how they combine with self-attention and a simple feed-forward network to create a complete, stable, and powerful Transformer Block. This block is the fundamental Lego brick that we will stack to build the full GPT model.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import math
import graphviz

# Set a seed for reproducibility
torch.manual_seed(42)

### The Core Idea: A Well-Run Committee Meeting

Imagine a token embedding as a "proposal" entering a committee meeting. The goal of the meeting is to refine this proposal by incorporating feedback from other proposals. A Transformer Block is like one round of this very structured meeting.

A naive meeting might have two simple steps:
1.  **Discussion (Self-Attention):** Every proposal "looks" at the other proposals to gather context and update itself. This is the communication phase.
2.  **Thinking (Feed-Forward Network):** After the discussion, each proposal is processed independently to think through the new information. This is the computation phase.

However, this naive process has problems. In a large meeting, some proposals might be "shouted" (large numerical values), drowning out others. This can make the entire process unstable, with values exploding or vanishing. Furthermore, a bad discussion round could completely corrupt a good starting proposal.

This is where two crucial components come in, acting as moderators:

1.  **Layer Normalization:** This is the meeting's chairperson. Before both the "discussion" and "thinking" phases, the chairperson says, "Everyone, please present your idea with the same volume." It forces all proposals to have a standard distribution (mean of 0, variance of 1). This prevents any single proposal from numerically dominating the process, leading to a much more stable and predictable meeting (and training process).

2.  **Residual Connections (or "Skip Connections"):** This is like a rule that says, "The output of any phase is just an *update* to the original proposal, not a replacement." We take the output of the attention or thinking phase and *add* it to the original input. This creates a shortcut, ensuring that the original information is never lost. Even if a phase is not helpful, we still retain the original proposal, and critically, this provides a direct path for learning signals (gradients) to flow backward through the network without vanishing.

So, a full Transformer Block isn't just `Attention -> MLP`. It's a carefully moderated process:

`Original Proposal + Discuss(Normalize(Original Proposal))`
`-> Result + Think(Normalize(Result))`

This structure ensures that each block refines the input in a stable and controlled way, allowing us to stack many of them together to form a deep, powerful model.

In [ ]:
# --- Mock Implementations of Components ---
# These are simplified for clarity. The previous notebook built a real CausalSelfAttention.

class CausalSelfAttention(nn.Module):
    """ A simplified stub for the Causal Self-Attention module. """
    def __init__(self, n_embd, n_head):
        super().__init__()
        # In a real implementation, this would contain query, key, value projections,
        # multi-head logic, and a causal mask. Here, we just simulate the output shape.
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        # For simplicity, we don't implement the actual attention logic here.
        # We just want a module that respects the input/output shapes.

    def forward(self, x):
        # This is a placeholder for the complex logic of attention.
        # It just projects the input and returns a tensor of the same shape.
        return self.c_proj(F.gelu(self.c_attn(x))) # Simplified transformation

class MLP(nn.Module):
    """ A simple Multi-Layer Perceptron. """
    def __init__(self, n_embd):
        super().__init__()
        self.c_fc    = nn.Linear(n_embd, 4 * n_embd)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * n_embd, n_embd)

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))

# --- The Main Attraction: The Transformer Block ---

class Block(nn.Module):
    """ The complete Transformer Block. """
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd)

    def forward(self, x):
        # First sub-layer: Attention with pre-normalization and a residual connection
        attention_output = self.attn(self.ln_1(x))
        x = x + attention_output

        # Second sub-layer: MLP with pre-normalization and a residual connection
        mlp_output = self.mlp(self.ln_2(x))
        x = x + mlp_output
        return x

### Walkthrough of the Implementation

Let's break down the `Block` class piece by piece.

**`__init__(self, n_embd, n_head)`**

This is the constructor where we define the building blocks.
-   `self.ln_1 = nn.LayerNorm(n_embd)`: This is the first "moderator". It will normalize the input tensor *before* it goes into the attention mechanism. We use PyTorch's built-in `LayerNorm`.
-   `self.attn = CausalSelfAttention(n_embd, n_head)`: This is our communication module. We're using the simplified stub we defined earlier.
-   `self.ln_2 = nn.LayerNorm(n_embd)`: The second "moderator". It normalizes the output of the first sub-layer (the attention + residual connection) before it's fed into the MLP.
-   `self.mlp = MLP(n_embd)`: The computation module, our simple feed-forward network.

**`forward(self, x)`**

This method defines the data flow, which is the heart of the Transformer Block.
-   `attention_output = self.attn(self.ln_1(x))`:
    - First, the input `x` is passed through `ln_1` to be normalized.
    - The clean, normalized tensor is then fed into the `attn` module.
-   `x = x + attention_output`:
    - This is the crucial **residual connection**. We take the output from the attention mechanism and *add* it back to the *original*, unmodified input `x`. The result is assigned back to `x`, becoming the input for the next stage.
-   `mlp_output = self.mlp(self.ln_2(x))`:
    - The new `x` (which now contains information from the attention step) is passed through the second normalizer, `ln_2`.
    - This normalized tensor is then fed into the `mlp` for independent processing.
-   `x = x + mlp_output`:
    - The second residual connection. The output of the MLP is added back to its input (`x`).
-   `return x`:
    - The final output tensor has the exact same shape as the input tensor, but its values have been refined by one block's worth of communication and computation. This consistent shape is what allows us to stack these blocks seamlessly.

In [ ]:
# --- Demonstration: A Single Block in Action ---

# Configuration
batch_size = 4        # How many independent sequences to process
context_length = 16   # The length of each sequence
n_embd = 32           # The dimensionality of each token embedding
n_head = 4            # Number of attention heads (for our mock attention)

# Create a sample input tensor
# (Batch, Time/Sequence, Features/Embedding)
input_tokens = torch.randn(batch_size, context_length, n_embd)

# Instantiate the block
transformer_block = Block(n_embd=n_embd, n_head=n_head)

# Pass the input through the block
output_tokens = transformer_block(input_tokens)

# Print the shapes to verify
print("Shape of input: ", input_tokens.shape)
print("Shape of output:", output_tokens.shape)

# The key takeaway is that the shape remains unchanged.
# The block takes a set of token embeddings and returns a refined set of the same size.
assert input_tokens.shape == output_tokens.shape

### Going Deeper: Pre-LN vs. Post-LN

A subtle but critical design choice in our `Block` is *where* we place the Layer Normalization. We used a "Pre-LN" formulation: `x + Sublayer(LayerNorm(x))`.

The original "Attention Is All You Need" paper actually used "Post-LN": `LayerNorm(x + Sublayer(x))`.

So why did nanoGPT (and GPT-2, GPT-3, and most modern Transformers) switch to Pre-LN?

**Stability.**

In Post-LN, the output of the residual connection is fed directly into the next block *without* being normalized. As you stack many blocks, the values along this main "residual path" can accumulate and grow very large. This makes the model very sensitive to the learning rate and often requires a "learning rate warmup" period (starting with a very small learning rate and gradually increasing it) to prevent the training from exploding at the beginning.

In Pre-LN, the input to every sub-layer (Attention and MLP) is *always* normalized. This acts like a control gate, ensuring that the operations inside the block receive well-behaved, nicely scaled inputs. The result is a much smoother gradient flow and more stable training, especially for very deep models. This small architectural change often removes the need for learning rate warmup and makes the overall training process more robust. It's a perfect example of how small implementation details can have a huge impact on large-scale model training.

In [ ]:
# --- Visualization of the Transformer Block ---

# This code uses the `graphviz` library to draw a diagram of the data flow.
dot = graphviz.Digraph(comment='Transformer Block Architecture')
dot.attr('node', shape='box', style='rounded,filled', fillcolor='lightblue')
dot.attr(rankdir='TB') # Top to Bottom layout

# Define nodes for the components
dot.node('Input', 'Input\n(B, T, C)')
dot.node('LN1', 'LayerNorm')
dot.node('Attn', 'Causal Self-Attention')
dot.node('Add1', '+', shape='circle', fillcolor='lightgreen')
dot.node('LN2', 'LayerNorm')
dot.node('MLP', 'MLP')
dot.node('Add2', '+', shape='circle', fillcolor='lightgreen')
dot.node('Output', 'Output\n(B, T, C)')

# Define the main data flow
dot.edge('Input', 'LN1')
dot.edge('LN1', 'Attn')
dot.edge('Attn', 'Add1')

# Define the first residual connection
dot.edge('Input', 'Add1', label=' Residual Path', style='dashed', constraint='false')

dot.edge('Add1', 'LN2')
dot.edge('LN2', 'MLP')
dot.edge('MLP', 'Add2')

# Define the second residual connection
dot.edge('Add1', 'Add2', label=' Residual Path', style='dashed', constraint='false')

dot.edge('Add2', 'Output')

# Render the graph
dot

### Summary & What's Next

In this notebook, we assembled the fundamental building block of a GPT model. We saw that a Transformer Block is much more than just an attention mechanism; it's a carefully designed unit that balances communication (attention) with computation (MLP), all while ensuring stability through normalization and preserving information with residual connections.

**Key Takeaways:**

*   **Two Sub-Layers:** A Transformer Block consists of two main parts: a self-attention layer and a feed-forward network (MLP).
*   **Layer Normalization is Crucial:** Applying LayerNorm *before* each sub-layer (Pre-LN) is key to stabilizing deep networks by ensuring the inputs to each operation are well-scaled.
*   **Residual Connections are the Superhighway:** The `x + ...` skip connections allow gradients and information to flow directly through the network, preventing them from vanishing and ensuring that each block only needs to learn how to *refine* the representation.
*   **Shape In, Shape Out:** A block processes a tensor of shape `(B, T, C)` and outputs a tensor of the exact same shape, making the blocks easily stackable.

We now have the Lego brick. In the next notebook, **"Assembling the Full GPT Model Architecture,"** we will take this `Block` and stack multiple copies of it together. We'll also add the necessary input (token and positional embeddings) and output (language modeling head) layers to construct the complete, end-to-end `GPT` model.